In [1]:
import pandas as pd
import numpy as np


import csv
import json
import requests


# Task 1.1

In [2]:
def parse_row(row):
    '''Parse a row from the CSV file and converts age, fee_inr and prior_no_shows 
    to Integers and distance_km to Float. Returning the parsed row and None if successful, 
    else returning None and the original row if there is a ValueError.'''
    
    try:        #Checking for ValueError while converting the data types of the columns
        row['age'] = int(row['age'])        
        row['fee_inr'] = int(row['fee_inr'])
        row['prior_no_shows'] = int(row['prior_no_shows'])
        row['distance_km'] = float(row['distance_km'])
        return row, None            #returning the parsed row and None if successful
    except ValueError as e:         #Handling ValueError
        return None, row            #returning None and the original row if there is a ValueError

In [3]:
# Reading the appointments.csv file
with open ('data/appointments.csv', 'r', newline="") as appointments_file:  
    reader = csv.DictReader(appointments_file)          #converting the CSV file to Dictionary format
    
    # reading the header of the appointments.csv file
    header = reader.fieldnames
    print(header)
    
    Value_count = 0         #variable to count the number of rows in the CSV file
    accepted = []           #list to store the accepted rows
    rejected = []           #list to store the rejected rows
    
    #Iterating through each row in the CSV file and parsing it using the parse_row function
    for row in reader:
        Value_count += 1
        validate_a, validate_r = parse_row(row)         #Calling the parse_row function to validate the data types of the columns

        #Appending the accepted row to accepted list and rejected row to rejected list
        if validate_a is not None:          
            accepted.append(validate_a)
        if validate_r is not None:
            rejected.append(validate_r)

    print(f'Row Count in file: {Value_count}')          #Printing the total number of rows present in the CSV file


print(f'Accepted Row Count: {len(accepted)}')           #Printing the total number of accepted rows after parsing the CSV file
print(f'Rejected Row Count: {len(rejected)}')           #Printing the total number of rejected rows after parsing the CSV file

['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Row Count in file: 1020
Accepted Row Count: 988
Rejected Row Count: 32


In [4]:
# Storing the Rejected Rows into a Json file
with open('rejected_rows.json', 'w')as rejected_file:
    json.dump(rejected, rejected_file, indent=2)

# Task 1.2

In [5]:
# Reading the appointments.csv file into a pandas DataFrame
appointment = pd.read_csv('data/appointments.csv')
appointment.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status
0,A0001,P509,C05,2025-01-24,2025-01-25,60.0,F,18.0,850,Phone,Consultation,Yes,1,No-Show
1,A0002,P046,C07,2025-05-27,2025-06-06,39.0,M,12.5,350,Walk-in,Follow-up,No,2,Attended
2,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
3,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
4,A0004,P482,C01,2025-04-18,2025-04-22,52.0,F,7.5,750,App,Consultation,Yes,0,Attended


In [6]:
print(f'Shape of the appointment DataFrame (Rows, Columns): {appointment.shape}')

Shape of the appointment DataFrame (Rows, Columns): (1020, 14)


In [7]:
# Displaying the information of the appointment DataFrame
appointment.info()

<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 111.7 KB


In [8]:
# Checking for missing or Null values in the appointment DataFrame
print('Number of Missing or Null values in each columns:')
appointment.isnull().sum()

Number of Missing or Null values in each columns:


appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           1
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              4
dtype: int64

In [9]:
# Checkiing for duplicate rows in the appointment DataFrame
print(f'Number of Duplicate rows in appointment DataFrame: {appointment.duplicated().sum()}')

Number of Duplicate rows in appointment DataFrame: 20


In [10]:
# Dropping the duplicate rows in the appointment DataFrame
appointment.drop_duplicates(inplace = True)

print(f'Total number of rows left after removing duplicates: {appointment.shape[0]}')

Total number of rows left after removing duplicates: 1000


In [11]:
# Checking the differnet values in `booking_channel` column 
appointment['booking_channel'].value_counts()

booking_channel
App         358
Phone       342
Walk-in     230
APP          10
Walk-in      10
app           9
PHONE         7
 walk-in      5
Walk-In       5
 App          5
phone         5
App           5
 phone        4
WALK-IN       3
Phone         1
walk-in       1
Name: count, dtype: int64

In [12]:
# Standardizing the values of `booking_channel` column
appointment['booking_channel'] = appointment['booking_channel'].str.strip().str.title()

appointment['booking_channel'].value_counts()

booking_channel
App        387
Phone      359
Walk-In    254
Name: count, dtype: int64

**Converting `appt_date` and `booked_on` column from str data type to datetime data type**

In [13]:
# For appt_date column
appointment['appt_date'] = pd.to_datetime(appointment['appt_date'], format='%Y-%m-%d', errors = 'coerce')
# For booked_on column
appointment['booked_on'] = pd.to_datetime(appointment['booked_on'], format = '%Y-%m-%d', errors = 'coerce')

# Reviewing the changes to the columns
appointment.info()

<class 'pandas.DataFrame'>
Index: 1000 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   appointment_id   1000 non-null   str           
 1   patient_id       1000 non-null   str           
 2   clinic_id        1000 non-null   str           
 3   booked_on        1000 non-null   datetime64[us]
 4   appt_date        994 non-null    datetime64[us]
 5   age              988 non-null    float64       
 6   gender           1000 non-null   str           
 7   distance_km      980 non-null    float64       
 8   fee_inr          1000 non-null   int64         
 9   booking_channel  1000 non-null   str           
 10  visit_type       1000 non-null   str           
 11  reminder_sent    1000 non-null   str           
 12  prior_no_shows   1000 non-null   int64         
 13  status           996 non-null    str           
dtypes: datetime64[us](2), float64(2), int64(2), str(8)
memor

In [14]:
# Rows that have Null values in `appt_date` column
print(f'Total rows with null appointment dates: {appointment["appt_date"].isnull().sum()}')

appointment[appointment['appt_date'].isnull() == True]

Total rows with null appointment dates: 6


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status
98,A0094,P487,C08,2025-02-24,NaT,53.0,M,6.5,850,Phone,Consultation,No,1,No-Show
167,A0162,P002,C02,2025-06-26,NaT,39.0,M,9.0,450,Phone,Follow-up,Yes,0,Attended
303,A0296,P497,C08,2025-03-04,NaT,69.0,M,5.4,600,App,Consultation,Yes,0,Attended
504,A0494,P401,C03,2025-05-29,NaT,34.0,F,8.1,350,App,Follow-up,No,0,Attended
534,A0523,P402,C02,2025-01-24,NaT,26.0,F,4.7,550,App,Follow-up,Yes,0,Attended
678,A0666,P018,C03,2025-06-21,NaT,71.0,F,2.2,450,Phone,Follow-up,Yes,0,Attended


In [15]:
# Removing the Null values present in `appt_date` column
appointment.dropna(subset=['appt_date'], inplace = True)

print(f'Total number of rows left after cleaning the column `appt_date`: {appointment.shape[0]}')       # 6 rows were removed.

Total number of rows left after cleaning the column `appt_date`: 994


In [16]:
# Rows that have Null values in `status` column
print(f'Total number of rows with null status: {appointment['status'].isnull().sum()}')

appointment[appointment['status'].isnull() == True]

Total number of rows with null status: 4


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status
199,A0194,P367,C01,2025-03-25,2025-04-10,39.0,F,3.5,700,Phone,Consultation,No,0,NaN
445,A0436,P228,C01,2025-03-05,2025-03-08,68.0,F,3.6,700,Walk-In,Consultation,Yes,1,NaN
562,A0551,P520,C04,2024-11-29,2025-01-02,54.0,F,16.1,650,App,Consultation,Yes,0,NaN
739,A0725,P376,C03,2025-05-27,2025-06-02,20.0,F,3.2,400,App,Follow-up,Yes,0,NaN


In [17]:
# Removing the Null Values present in `status` column
appointment.dropna(subset=['status'], inplace = True)

print(f'Total number of rows left after cleaning the column `status`: {appointment.shape[0]}')      # 4 rows were removed

Total number of rows left after cleaning the column `status`: 990


# Task 1.3

In [18]:
# Number of Null values before converting invalid ages to Null
appointment.isnull().sum()

appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           0
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              0
dtype: int64

In [19]:
# Converting Invalid ages (i.e. less than 1 or greater than 100) to Null values
appointment.loc[
    (appointment["age"] < 1) | (appointment["age"] > 100), "age"
] = None

In [20]:
# Number of Null values after converting invalid ages to Null
appointment['age'].isnull().sum()

np.int64(17)

In [21]:
# Inserting the age column's median to fill the Null values  
appointment['age'] = appointment['age'].fillna(round(appointment['age'].median())).astype(int)

In [22]:
appointment['age'].isnull().sum()       # Checking for any left out Null values in age column

np.int64(0)

In [23]:
# Inserting the distance_km column's median to fill the Null values  
appointment['distance_km'] = appointment['distance_km'].fillna(appointment['distance_km'].median())

In [24]:
# Final check for the Null values
appointment.isnull().sum()

appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
dtype: int64

In [25]:
# Defining a function which calculates IQR and rcaps outliers
def clip(X):
    # Calculating quantile
    q1= np.quantile(X, 0.25)
    q3= np.quantile(X, 0.75)

    iqr = q3-q1     # Calculating IQR

    # Calculating lower and upper bond
    l_fence = q1 - 1.5*iqr      
    u_fence = q3 + 1.5*iqr

    # Counting the number of Outliers
    upper_clipped = appointment[X > u_fence].shape[0]
    lower_clipped = appointment[X < l_fence].shape[0]

    # Capping the Outliers
    if X.name == 'distance_km':
        series_capped = round(np.clip(X,l_fence,u_fence),1)
    else:
        series_capped = round(np.clip(X,l_fence,u_fence),0).astype(int)

    return lower_clipped, upper_clipped, l_fence ,u_fence ,series_capped

In [26]:
# Capping values for distance_km column
low, up, low_val, up_val, appointment['distance_km'] = clip(appointment['distance_km'])

print(f'the number of values for the `distance_km` column that were capped \n below the value {low_val} : {low} \n and above the value {up_val} : {up}')

the number of values for the `distance_km` column that were capped 
 below the value -7.9375 : 0 
 and above the value 25.5625 : 26


In [27]:
# Capping values for distance_km column
low, up, low_val, up_val, appointment['fee_inr'] = clip(appointment['fee_inr'])

print(f'the number of values for the `fee_inr` column that were capped \n below the value {low_val} : {low} \n and above the value {up_val} : {up}')

the number of values for the `fee_inr` column that were capped 
 below the value 68.75 : 0 
 and above the value 1218.75 : 3


# Task 1.4

In [28]:
# Creating a new column called lead_days from the difference of appt_date and booked_on columns
appointment['lead_days'] = appointment['appt_date'] - appointment['booked_on']
appointment['lead_days'] = appointment['lead_days'].dt.days

appointment.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6
4,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4
5,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40


In [29]:
# Creating a new column no_show 
appointment['no_show'] = (appointment['status'] == 'No-Show').astype(int)

print(f"Total No Show registered {appointment['no_show'].sum()} out of {appointment.shape[0]} giving: {(appointment['no_show'].sum()/appointment.shape[0])*100:.2f} percentage")
appointment.head()

Total No Show registered 218 out of 990 giving: 22.02 percentage


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
4,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
5,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [30]:
# Creating a numpy from the age, distance_km, fee_inr, lead_days and printing their mean value
X = appointment[["age","distance_km","fee_inr","lead_days"]].to_numpy()

X.mean(axis = 0)

array([ 48.17373737,   9.50090909, 635.71414141,  11.81919192])

In [31]:
print(f'Mean of the columns\n age: {X[:,0].mean():.2f}\n distance_km: {X[:,1].mean():.2f}\n fee_inr: {X[:,2].mean():.2f}\n lead_days: {X[:,3].mean():.2f}')

Mean of the columns
 age: 48.17
 distance_km: 9.50
 fee_inr: 635.71
 lead_days: 11.82


In [32]:
# Calculating the number of appointment that were booked before 14 days
mask = X[:,3] > 14

print(f'The Number of appointments that were booked before 14 days {mask.sum()} with percentage: {(mask.mean())*100:.2f}')

The Number of appointments that were booked before 14 days 296 with percentage: 29.90


In [33]:
# Calculating the Z-score of distance_km column and counting the valued 
z_distance = (X[:,1] - np.mean(X[:,1])) / np.std(X[:,1])

print(f"Values that are statistically significant outliers, as they sit beyond the 95% confidence interval are: {(np.abs(z_distance) > 2).sum()}")

Values that are statistically significant outliers, as they sit beyond the 95% confidence interval are: 53


# Task 1.5

In [34]:
# Reading clinics.csv
clinics = pd.read_csv('data/clinics.csv')

clinics

,clinic_id,clinic_name,city,tier,latitude,longitude
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462


In [35]:
# Adding temp_c column to the clinics csv and saving it as clinics_enriched.csv
for i in range(len(clinics)):
    params={"latitude": clinics['latitude'][i], "longitude": clinics['longitude'][i], "hourly": "temperature_2m", "forecast_days": 1}
    response = requests.get("https://api.open-meteo.com/v1/forecast", params = params)
    if response.status_code == 200:
        data = response.json()
        temp_c = data["hourly"]["temperature_2m"][0]
        clinics.loc[i,'temp_c'] = temp_c

clinics.to_csv("clinics_enriched.csv", index=False)


# Task 1.6

In [36]:
# Converting appt_date and booked_on from datetime to string
appointment['appt_date'] = appointment['appt_date'].dt.strftime("%Y-%m-%d")
appointment['booked_on'] = appointment['booked_on'].dt.strftime("%Y-%m-%d")

appointment.shape

(990, 16)

In [37]:
# soring the appointment table on appointment_id column and saving it to appointment_clean.csv file
appointment = appointment.sort_values(by='appointment_id')

appointment.to_csv('appointments_clean.csv',index=False)

# Creating a new file containing inserting values of clinics_enriched.csv and appointment_clean.csv for SQL file


In [38]:
with open("load_data.sql", "w") as f:

    # Clinics first
    for _, row in clinics.iterrows():
        f.write(
            f"INSERT INTO clinics VALUES "
            f"('{row['clinic_id']}','{row['clinic_name']}','{row['city']}','{row['tier']}','{row['latitude']}','{row['longitude']}','{row['temp_c']}');\n"
        )

    # Appointments second
    for _, row in appointment.iterrows():
        f.write(
            f"INSERT INTO appointments VALUES "
            f"('{row['appointment_id']}','{row['patient_id']}','{row['clinic_id']}',"
            f"'{row['booked_on']}','{row['appt_date']}',"
            f"{row['age']},"
            f"'{row['gender']}',"
            f"{row['distance_km']},"
            f"{row['fee_inr']},"
            f"'{row['booking_channel']}',"
            f"'{row['visit_type']}',"
            f"'{row['reminder_sent']}',"
            f"{row['prior_no_shows']},"
            f"'{row['status']}',"
            f"{row['lead_days']},"
            f"{row['no_show']}"
            f");\n"
        )